# Logistic Regression Training (PySpark ML)

Train logistic regression models for network intrusion detection using PySpark ML to handle large datasets (10GB+).

## Strategies:
1. **Balanced Data**: Train on oversampled balanced dataset
2. **Class Weighting**: Use class weights to handle imbalance

In [1]:
import os
import sys
sys.path.append('..')

from pathlib import Path
from notebooks.training_utils import (
    load_training_data_pyspark,
    train_and_evaluate_pyspark,
    save_models_pyspark,
    print_summary_pyspark
)

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

# Stop any existing Spark session
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("Stopped existing Spark session")
except:
    pass

# Create Spark session optimized for ML training
spark = SparkSession.builder \
    .appName("LogisticRegressionTraining") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .config("spark.default.parallelism", "10") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"✓ Spark initialized")
print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/17 20:49:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ Spark initialized
Spark Version: 3.5.6
Spark UI: http://mac.home:4041


26/01/17 20:49:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Load Data

In [2]:
# Load data using utility function
train_orig_vec, train_balanced_vec, test_vec, feature_cols, project_root = load_training_data_pyspark(spark)

Loading training and test data from Parquet with PySpark...
✓ Data loaded
  Original train: 4,106,722 rows, 333 features
  Balanced train: 7,756,447 rows, 333 features
  Test: 1,028,707 rows, 333 features
✓ Combined features and labels
✓ Assembled feature vectors

Original train class distribution:


ERROR:root:KeyboardInterrupt while sending command.               (2 + 10) / 14]
Traceback (most recent call last):
  File "/Users/matthewweaver/Repositories/nidstream/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/matthewweaver/Repositories/nidstream/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/matthewweaver/.local/share/uv/python/cpython-3.11.13-macos-aarch64-none/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Data is already prepared by the utility function
print(f"✓ Ready for training with {len(feature_cols)} features")

## 2. Train Models

In [ ]:
# Strategy 1: Balanced Data
model_params = {
    'featuresCol': 'features',
    'labelCol': 'label',
    'maxIter': 100,
    'regParam': 0.01,
    'elasticNetParam': 0.0,  # L2 regularization
    'family': 'binomial'
}

model_balanced, metrics_balanced, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_balanced_vec,
    test_vec,
    "Logistic Regression - Balanced Data Strategy",
    use_class_weights=False
)

In [ ]:
# Strategy 2: Class Weighting
model_weighted, metrics_weighted, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_orig_vec,
    test_vec,
    "Logistic Regression - Class Weight Strategy",
    use_class_weights=True
)

## 3. Save Models

In [ ]:
# Save models and metrics
save_models_pyspark(model_balanced, model_weighted, metrics_balanced, metrics_weighted, 'lr', project_root)

## 4. Summary

In [ ]:
# Print comparison summary
print_summary_pyspark(metrics_balanced, metrics_weighted, "Logistic Regression")

In [ ]:
# Stop Spark session when done
# spark.stop()